## Load libraries and datasets for proteomic and sigma network analyses

In [1]:
import pandas as pd
import numpy as np
import math
import os.path

In [2]:
#The sigma_network dataset is sourced from RegulonDB, version 12.
proteomic_cond = 'infiles/allgen4691_fitfg_newname.csv'
sigma_network = pd.read_csv('infiles/NetworkSigmaGene.tsv', sep='\t',  comment='#')
proteomic_cond =  pd.read_csv(proteomic_cond)
sigma_network.rename(columns={"1)sigmaName": "Sigma", "2)regulatedName": "Target", "3)function": "Regulation", 
                              "5)confidenceLevel ": "confidence"}, inplace=True)
snetwork = sigma_network.drop(['4)promoterEvidence'], axis=1)
snetwork

,Sigma,Target,Regulation,confidence
0,sigma19,fecA,+,C
1,sigma19,fecB,+,C
2,sigma19,fecC,+,C
3,sigma19,fecD,+,C
4,sigma19,fecE,+,C
...,...,...,...,...
2576,sigma70,ytiD,+,S
2577,sigma70,zinT,+,S
2578,sigma70,zntA,+,S
2579,sigma70,znuB,+,S


### Select relevant columns from the proteomic dataset and remove duplicates in the "Target" column.

In [3]:
#Select columns of interest related to gene identifiers (bnum, gene) and protein expression levels 
#under different carbon sources (e.g., Glucose, LB, Glucosamine, etc.).
protein_fg = proteomic_cond [["bnum","gene","Glucose","LB","Glucosamine","Glycerol","Xylose","Mannose","Galactose","Fructose","Acetate","Pyruvate","Succinate"]]
protein_fg = protein_fg.copy()
protein_fg.rename(columns={"gene":"Target"}, inplace=True, copy=False)
protein_fg
#Filter rows to exclude any entries where "Target" (gene name) is null or missing.
df2 = protein_fg[protein_fg.Target.notnull()]
prot_fgs = df2.drop_duplicates(subset='Target', keep="first").reset_index(drop = True)
prot_fgs

,bnum,Target,Glucose,LB,Glucosamine,Glycerol,Xylose,Mannose,Galactose,Fructose,Acetate,Pyruvate,Succinate
0,b4053,alr,0.000946,0.000945,0.000756,0.000475,0.001033,0.000475,0.000531,0.000798,0.001104,0.000853,0.000606
1,b0764,modB,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,b2413,cysZ,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,b3639,dfp,0.018157,0.042918,0.024835,0.017690,0.019203,0.016354,0.013236,0.020423,0.016088,0.030884,0.013245
4,b4123,dcuB,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4685,b4841,ascrp,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4686,b4842,asompR,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4687,b4843,asphoP,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4688,b4844,ameF,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [4]:
#Merge the sigma network and proteomic datasets based on the "Target" column and 
#fill any missing values from prot_fgs with 0, as a placeholder for cases where no proteomic data is available.
net_and_conds = pd.merge(snetwork, prot_fgs, on="Target", how="left").fillna(0)
net_and_conds.head()

,Sigma,Target,Regulation,confidence,bnum,Glucose,LB,Glucosamine,Glycerol,Xylose,Mannose,Galactose,Fructose,Acetate,Pyruvate,Succinate
0,sigma19,fecA,+,C,b4291,0.000222,0.000499,0.000247,0.000274,0.000896,0.000641,0.000458,0.00017,0.000582,0.000568,0.000528
1,sigma19,fecB,+,C,b4290,0.000600,0.029732,0.000618,0.000151,0.000000,0.000000,0.000148,0.00000,0.000226,0.000384,0.000199
2,sigma19,fecC,+,C,b4289,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
3,sigma19,fecD,+,C,b4288,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
4,sigma19,fecE,+,C,b4287,0.000645,0.011063,0.000356,0.000285,0.000000,0.000000,0.000065,0.00000,0.000081,0.000224,0.000215


In [5]:
#Count occurrences of each unique gene target in the merged dataset.
df3 = net_and_conds.groupby(['Target'])["Target"].count()
df3

Target
3'ETS-<i>leuZ</i>    1
accA                 1
accB                 1
accC                 1
accD                 1
                    ..
znuB                 1
znuC                 1
zraP                 1
zraR                 1
zraS                 1
Name: Target, Length: 2152, dtype: int64

In [6]:
# Identify and count unique gene targets in the merged dataset.
n=0
gen_sigs = np.unique(net_and_conds["Target"])
print("Total target:", len(gen_sigs))

Total target: 2152


### Summarize the data by sigma factors and merge counts with total glucose values.

In [7]:
#Sum PL by sigma, only under the Glucose condition
suma_sigma = net_and_conds.groupby('Sigma')['Glucose'].sum().reset_index()
sig_counts = net_and_conds['Sigma'].value_counts().reset_index()
sig_counts.columns = ['Sigma', 'Counts']
suma_sigma.columns = ['Sigma', 'Total_PL']
tbl_sig = sig_counts.merge(suma_sigma, on="Sigma", how="left")
tbl_sig

,Sigma,Counts,Total_PL
0,sigma70,1710,168.366324
1,sigma38,340,27.962007
2,sigma32,177,15.156013
3,sigma54,141,3.674022
4,sigma24,132,27.006488
5,sigma28,76,1.854359
6,sigma19,5,0.001467


### Calculate and print the proteome fraction of "rpoS" for each condition, store the results in a list.

In [8]:
condition = ['Glucose','LB','Glucosamine','Glycerol','Xylose','Mannose','Galactose','Fructose','Acetate','Pyruvate','Succinate']
pf = []

for c in condition:
    suma_sigma = net_and_conds.groupby('Sigma')[c].sum().reset_index()
    sig_counts = net_and_conds['Sigma'].value_counts().reset_index()
    sig_counts.columns = ['Sigma', 'Counts']
    suma_sigma.columns = ['Sigma', 'Total_PL']
    tbl_sig = sig_counts.merge(suma_sigma, on="Sigma", how="left")
    total_proteome = net_and_conds[c].sum()
#    print(c, total_proteome)
    
    rpoS = tbl_sig.loc[1]
    proteome_fraction = (rpoS.Total_PL / total_proteome) * 100
    pf.append(proteome_fraction) 
    
# Create a DataFrame to store the condition and corresponding proteome fractions.
df = pd.DataFrame(list(zip(condition, pf)), columns=['condition','proteome fraction'])
df = df.sort_values(by='proteome fraction', ascending=False)
df

,condition,proteome fraction
6,Galactose,15.240666
7,Fructose,13.690263
2,Glucosamine,13.335819
4,Xylose,13.137668
9,Pyruvate,13.010003
8,Acetate,12.370058
5,Mannose,12.365210
3,Glycerol,12.018742
10,Succinate,11.718973
0,Glucose,11.458868


### Filter the net_and_conds DataFrame to retrieve rows corresponding to the sigma factor 'sigma38'.

In [9]:
net_and_conds.loc[net_and_conds['Sigma']=='sigma38'] 

,Sigma,Target,Regulation,confidence,bnum,Glucose,LB,Glucosamine,Glycerol,Xylose,Mannose,Galactose,Fructose,Acetate,Pyruvate,Succinate
390,sigma38,aceE,+,C,b0114,0.961555,1.954612,0.694257,0.672533,1.693791,0.811060,0.539617,1.556716,0.404364,2.611338,0.783641
391,sigma38,aceF,+,C,b0115,0.993200,2.449843,0.624556,0.619484,1.494406,0.747221,0.535949,1.232343,0.391089,2.404707,0.755587
392,sigma38,acnA,+,S,b1276,0.096485,0.158014,0.247154,0.130967,0.081873,0.205091,0.212082,0.109441,0.259464,0.127278,0.191128
393,sigma38,acs,+,S,b4069,0.118983,0.280560,0.938723,0.474435,0.237714,1.208027,1.180795,0.297746,0.788234,0.656106,1.097680
394,sigma38,actP,+,S,b4067,0.003972,0.011103,0.037128,0.024339,0.005586,0.067326,0.066594,0.008610,0.032974,0.026410,0.062028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,sigma38,yqiG,+,W,b3046,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
726,sigma38,yqiH,+,W,b3047,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
727,sigma38,yqiI,+,W,b3048,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
728,sigma38,ytfK,+,S,b4217,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Create a DataFrame to store unique sigma factors and their corresponding target genes.

In [10]:
sig = np.unique(net_and_conds["Sigma"])
name_list = ['s24', 's28', 's32','s38', 's54', 's70']
set_sigs = []

for s in sig:
    sig_tar = list(net_and_conds[(net_and_conds["Sigma"] == s)].Target.values)
    set_sigs.append({
        'Sets': s,
        'genes': sig_tar
    })
tbl_sig2 = pd.DataFrame(set_sigs)
tbl_sig2["genes"] = tbl_sig2["genes"].apply(set)
tbl_sig2

,Sets,genes
0,sigma19,"{fecC, fecB, fecA, fecD, fecE}"
1,sigma24,"{asflhD, hpf, apaG, fkpA, mscM, sbmA, clpX, hc..."
2,sigma28,"{cheZ, fliX, recC, fliI, motA, ves, modC, ygdB..."
3,sigma32,"{lspA, nfuA, pncC, yafD, yccE, rrfD, pphA, rrl..."
4,sigma38,"{fau, mdtF, rssB, treF, rraA, mlrA, patD, sgrS..."
5,sigma54,"{flhD, actP, hypF, ddpC, potH, rutD, relA, psp..."
6,sigma70,"{rnd, atpE, acrF, cynT, cysG, purR, trpB, nrdD..."


In [11]:
tbl_sig = tbl_sig2.T
tbl_sig

,0,1,2,3,4,5,6
Sets,sigma19,sigma24,sigma28,sigma32,sigma38,sigma54,sigma70
genes,"{fecC, fecB, fecA, fecD, fecE}","{asflhD, hpf, apaG, fkpA, mscM, sbmA, clpX, hc...","{cheZ, fliX, recC, fliI, motA, ves, modC, ygdB...","{lspA, nfuA, pncC, yafD, yccE, rrfD, pphA, rrl...","{fau, mdtF, rssB, treF, rraA, mlrA, patD, sgrS...","{flhD, actP, hypF, ddpC, potH, rutD, relA, psp...","{rnd, atpE, acrF, cynT, cysG, purR, trpB, nrdD..."


### Extract unique target genes associated with specific sigma factors.

In [12]:
# Retrieve the target genes for each specified sigma factor and store them in separate variables.
Sigma19 = tbl_sig2[tbl_sig2["Sets"] == "sigma19"]["genes"].values[0]
Sigma24 = tbl_sig2[tbl_sig2["Sets"] == "sigma24"]["genes"].values[0]
Sigma28 = tbl_sig2[tbl_sig2["Sets"] == "sigma28"]["genes"].values[0]
Sigma32 = tbl_sig2[tbl_sig2["Sets"] == "sigma32"]["genes"].values[0]
Sigma38 = tbl_sig2[tbl_sig2["Sets"] == "sigma38"]["genes"].values[0]
Sigma54 = tbl_sig2[tbl_sig2["Sets"] == "sigma54"]["genes"].values[0]
Sigma70 = tbl_sig2[tbl_sig2["Sets"] == "sigma70"]["genes"].values[0]
#usigma38 = set(sigma38)

#Calculate the unique genes associated with sigma38 that are not present in the target genes of other sigma factors.
us38= Sigma38.difference(Sigma19,Sigma24,Sigma32,Sigma54,Sigma70)
us38
print(len(us38))
lista_us38 =list(us38)

131


In [13]:
# filter to find entries with targets that are unique to sigma38.
sigma38_uniq = net_and_conds[net_and_conds["Target"].isin(lista_us38)]
sigma38_uniq

,Sigma,Target,Regulation,confidence,bnum,Glucose,LB,Glucosamine,Glycerol,Xylose,Mannose,Galactose,Fructose,Acetate,Pyruvate,Succinate
398,sigma38,aldB,+,S,b3588,0.054978,0.068118,0.107263,0.098507,0.055517,0.132017,0.273813,0.042521,0.159829,0.126904,0.110274
401,sigma38,ansP,+,S,b1453,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
422,sigma38,baeR,+,W,b2079,0.012673,0.008626,0.008838,0.008294,0.012478,0.008235,0.008979,0.009984,0.007537,0.006026,0.006961
423,sigma38,baeS,+,W,b2078,0.000791,0.001257,0.001033,0.001341,0.001123,0.001356,0.000759,0.001501,0.001758,0.000894,0.001125
424,sigma38,blc,+,S,b4149,0.001630,0.006397,0.002822,0.002330,0.001735,0.002497,0.002548,0.001664,0.002716,0.001574,0.002094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,sigma38,yqiG,+,W,b3046,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
726,sigma38,yqiH,+,W,b3047,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
727,sigma38,yqiI,+,W,b3048,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
728,sigma38,ytfK,+,S,b4217,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Display the resulting DataFrame containing conditions and their corresponding proteome fractions for sigma38.

In [14]:
# Calculate the proteome fraction for sigma38 over each condition
condition = ['Glucose','LB','Glucosamine','Glycerol','Xylose','Mannose','Galactose','Fructose','Acetate','Pyruvate','Succinate']
#condition = ['Galactose','Fructose','Glucosamine','Xylose','Pyruvate','Acetate','Mannose','Glycerol','Succinate','Glucose','LB']
pf130 = []

for c in condition:
    if c in sigma38_uniq.columns:
        column = sigma38_uniq[c]
        sum_sig130 = column.sum()        
        total_proteome = net_and_conds[c].sum()
        proteome_fraction2 = (sum_sig130 / total_proteome) * 100
        pf130.append(proteome_fraction2) 

df2 = pd.DataFrame(list(zip(condition, pf130)), columns=['condition','proteome fraction'])
df2 = df2.sort_values(by='proteome fraction', ascending=False)
#df2 = df2.sort_index()
df2

,condition,proteome fraction
6,Galactose,2.329051
2,Glucosamine,1.681953
8,Acetate,1.648632
5,Mannose,1.589529
3,Glycerol,1.579118
10,Succinate,1.443809
9,Pyruvate,1.264029
4,Xylose,1.245804
0,Glucose,1.213936
7,Fructose,1.124782


### Figure 06. Panels A-B

In [17]:
# Modified by David

fig = sp.make_subplots(rows=2, cols=1, subplot_titles=("All genes RpoS-dependent, n=340",
                                                       "Unique genes RpoS-dependent, n=131"))

# Add the first graph to the first subplot
fig.add_trace(go.Bar(
    x=df['condition'],
    y=df['proteome fraction'],
    marker_color=px.colors.qualitative.Vivid[7],
    showlegend=False
), row=1, col=1)

# Add the second graph to the second subplot
fig.add_trace(go.Bar(
    x=df2['condition'],
    y=df2['proteome fraction'],
    marker_color=px.colors.qualitative.Vivid[7],
    showlegend=False
), row=2, col=1)

fig.update_xaxes(tickangle=-60, tickvals=list(range(len(df['condition']))), ticktext=df['condition'],
                 tickfont=dict(size=16), row=1, col=1)
fig.update_xaxes(tickangle=-60, tickvals=list(range(len(df2['condition']))), ticktext=df2['condition'],
                 tickfont=dict(size=16), row=2, col=1)

fig.update_yaxes(title_text='Released proteome (%)', title_standoff=4, title_font=dict(size=16),
                 tickfont=dict(size=14), row=1, col=1)
fig.update_yaxes(title_text='Released proteome (%)', title_standoff=4, title_font=dict(size=16),
                 tickfont=dict(size=14), row=2, col=1)

fig.update_layout(
    height=800, 
    width=650, 
    showlegend=False,
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)"
)

fig.show()
fig.write_image("./Figures/Fig06_RpoS.svg")
fig.write_image("./Figures/Fig06_RpoS.png")

In [18]:
# Filter the `net_and_conds` DataFrame to create separate DataFrames for each sigma type.
# Each DataFrame contains rows where the 'Sigma' column matches the specified sigma value.
sigma19 = net_and_conds.loc[net_and_conds['Sigma']=='sigma19'] 
sigma24 = net_and_conds.loc[net_and_conds['Sigma']=='sigma24'] 
sigma28 = net_and_conds.loc[net_and_conds['Sigma']=='sigma28'] 
sigma32 = net_and_conds.loc[net_and_conds['Sigma']=='sigma32'] 
sigma38 = net_and_conds.loc[net_and_conds['Sigma']=='sigma38'] 
sigma54 = net_and_conds.loc[net_and_conds['Sigma']=='sigma54'] 
sigma70 = net_and_conds.loc[net_and_conds['Sigma']=='sigma70'] 

In [19]:
# Create a dictionary of sigma factors and their corresponding DataFrames
sigma_data = {
    'sigma 19': sigma19,
    'sigma 24': sigma24,
    'sigma 28': sigma28,
    'sigma 32': sigma32,
    'sigma 38': sigma38,
    'sigma 54': sigma54,
    'sigma 70': sigma70
}

# Extract the 'Target' column from each DataFrame, ensuring consistent length
data = {}
max_length = 0

# Determine the maximum length among all lists
for key, df in sigma_data.items():
    if 'Target' in df.columns:  
        data[key] = df['Target'].tolist()  
        max_length = max(max_length, len(data[key]))

# Pad shorter lists with NaN values to match the maximum length
for key in data.keys():
    data[key] += [np.nan] * (max_length - len(data[key]))

# Combine the lists into a single DataFrame
combined_df = pd.DataFrame(data)
combined_df = combined_df.dropna(how='all')
# combined_df.to_csv('tbl_sig.csv', index=False)
combined_df.head()

,sigma 19,sigma 24,sigma 28,sigma 32,sigma 38,sigma 54,sigma 70
0,fecA,ahpF,aer,ackA,aceE,acrD,3'ETS-<i>leuZ</i>
1,fecB,apaG,cheA,adiC,aceF,actP,accA
2,fecC,apaH,cheB,alaA,acnA,amtB,accB
3,fecD,asflhD,cheR,alaT,acs,argT,accC
4,fecE,bacA,cheW,alaU,actP,aslB,accD
